# 10 Batch MD Benchmark

This notebook is a launcher and tracker for the multi-system MD benchmark for small PHA oligomers. The benchmark checks whether the existing PHB4 workflow also works for related PHA oligomers with different side chains and lengths.

Run one system at a time to avoid long blocking notebook runs. Larger oligomers may take much longer during GAFF2/antechamber, so the reusable automation stays in `scripts/run_md_test_set.py`.

For each selected system, the batch script follows the same staged shape as the existing `examples/output/md_tests/PHB4/` tutorial output: `gaff2/`, `openmm/dry_polymer/`, `gromacs/dry_polymer/`, and `gromacs/solvated_polymer/`.

## Benchmark Systems

The test set contains the six systems below.

In [ ]:
systems = [
    "PHB4",
    "PHB8",
    "PHO4",
    "PHO8",
    "PHDD4",
    "PHDD8",
]

systems

## Selected System

Change this value before launching the script. Keep each run scoped to one system unless you intentionally want a longer batch job.

In [ ]:
selected_system = "PHB4"
if selected_system not in systems:
    raise ValueError(f"Unknown benchmark system: {selected_system}")


selected_system

## Output Folder Structure

Benchmark output is written under `examples/output/benchmark/`.

Expected system folders:

- `examples/output/benchmark/PHB4/`
- `examples/output/benchmark/PHB8/`
- `examples/output/benchmark/PHO4/`
- `examples/output/benchmark/PHO8/`
- `examples/output/benchmark/PHDD4/`
- `examples/output/benchmark/PHDD8/`

In [ ]:
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
md_tests_root = repo_root / "examples" / "output" / "benchmark"
expected_system_dirs = [md_tests_root / system for system in systems]

print(f"Repository: {repo_root}")
print(f"Benchmark output root: {md_tests_root}")
for path in expected_system_dirs:
    print(path.relative_to(repo_root))

## Run the Batch Script

The script performs the batch work. This notebook launches only `selected_system` by default so each benchmark run can finish or fail independently.

The `--prepare-gromacs` flag prepares the dry GROMACS folder and the `solvated_polymer` setup files, including `run_solvate_local.sh`, ion/water topology includes, minimisation/equilibration `.mdp` files, and HPC scripts. It does not run the local GROMACS solvation script.

In [ ]:
script_path = repo_root / "scripts" / "run_md_test_set.py"
!python3 "$script_path" --system "$selected_system" --prepare-gromacs

## Check Progress

These checks report progress across all six benchmark systems. They do not duplicate the workflow logic from the batch script.

In [ ]:
folder_status = {
    system: (md_tests_root / system).exists()
    for system in systems
}

folder_status

In [ ]:
progress_rows = []

for system in systems:
    system_root = md_tests_root / system
    expected_files = {
        "input_sdf": repo_root / "examples" / "output" / "polymer_structures" / f"{system}_R.sdf",
        "gaff2_prmtop": system_root / "gaff2" / f"{system}.prmtop",
        "gaff2_inpcrd": system_root / "gaff2" / f"{system}.inpcrd",
        "gaff2_mol2": system_root / "gaff2" / f"{system}.gaff2.mol2",
        "gaff2_frcmod": system_root / "gaff2" / f"{system}.gaff2.frcmod",
        "openmm_summary": system_root / "openmm" / "dry_polymer" / "openmm_summary.log",
        "openmm_final_pdb": system_root / "openmm" / "dry_polymer" / "final.pdb",
        "gromacs_dry_gro": system_root / "gromacs" / "dry_polymer" / "step5_input.gro",
        "gromacs_dry_top": system_root / "gromacs" / "dry_polymer" / "topol.top",
        "gromacs_solvated_gro": system_root / "gromacs" / "solvated_polymer" / "step5_input.gro",
        "gromacs_solvated_top": system_root / "gromacs" / "solvated_polymer" / "topol.top",
        "gromacs_solvate_script": system_root / "gromacs" / "solvated_polymer" / "run_solvate_local.sh",
        "gromacs_ions_mdp": system_root / "gromacs" / "solvated_polymer" / "ions.mdp",
        "gromacs_tip3p": system_root / "gromacs" / "solvated_polymer" / "TIP3_SOL.itp",
        "gromacs_sodium": system_root / "gromacs" / "solvated_polymer" / "SOD.itp",
        "gromacs_chloride": system_root / "gromacs" / "solvated_polymer" / "CLA.itp",
        "gromacs_nvt_mdp": system_root / "gromacs" / "solvated_polymer" / "step6.1_nvt.mdp",
        "gromacs_npt_mdp": system_root / "gromacs" / "solvated_polymer" / "step6.2_npt.mdp",
        "gromacs_production_mdp": system_root / "gromacs" / "solvated_polymer" / "step7_production.mdp",
    }
    progress_rows.append({
        "system": system,
        **{label: path.exists() for label, path in expected_files.items()},
    })

progress_rows

In [ ]:
for row in progress_rows:
    completed = sum(value is True for key, value in row.items() if key != "system")
    total = len(row) - 1
    print(f"{row['system']}: {completed}/{total} expected files present")